# 訓練行人號誌與斑馬線模型（盲人尋道）

只認三類：`ped_green`（小綠人）、`ped_red`（小紅人）、`crosswalk`（斑馬線）。原本偵測行人、車輛的模型不動。

**開始前**：
1. 上方選單 **執行階段 → 變更執行階段類型 → T4 GPU**。
2. 把 Roboflow 匯出的 zip 改名為 `signals-dataset.zip`，放到 Google 雲端硬碟的 `科展/` 資料夾。

然後從上到下一格一格執行。詳細的拍攝、標註方法見 `training/README.md`。

## 1. 設定

In [ ]:
# 要改的設定都在這裡
DRIVE_DIR = "/content/drive/MyDrive/科展"       # 雲端硬碟裡放資料集、存結果的資料夾
DATASET_ZIP = "signals-dataset.zip"             # Roboflow 匯出的 zip 檔名
EXPECTED_CLASSES = ["ped_green", "ped_red", "crosswalk"]   # 順序不能改：App 會照這個順序讀

IMGSZ = 320        # 跟 App 現在的偵測模型一樣大小；遠處的小號誌認不出來時可以試 416（App 也要跟著改）
EPOCHS = 100       # 最多訓練幾輪
PATIENCE = 20      # 連續這麼多輪沒有進步就提早停止
BATCH = 16         # T4 GPU 用 16；出現 out of memory 時改成 8
SEED = 0           # 固定隨機種子：同樣的資料重跑一次結果一樣，報告才能重現
OUTPUT_NAME = "signals-yolo11n-320.onnx"

## 2. 確認 GPU、安裝訓練套件

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "沒有 GPU：請到「執行階段 → 變更執行階段類型」選 T4 GPU"
!pip install -q "ultralytics>=8.3.0" onnxruntime

import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())
assert torch.cuda.is_available(), "沒有 GPU，訓練會非常慢：請先切換到 T4 GPU 再從頭執行"

## 3. 從雲端硬碟拿資料集

In [ ]:
import os, zipfile, shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
zip_path = Path(DRIVE_DIR) / DATASET_ZIP
assert zip_path.exists(), f"找不到 {zip_path}：請確認 zip 檔已經放到雲端硬碟的「科展」資料夾，檔名是 {DATASET_ZIP}"

DATA = Path("/content/dataset")
shutil.rmtree(DATA, ignore_errors=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(DATA)
# 有些 zip 多包了一層資料夾：找出 data.yaml 真正的位置
yaml_files = list(DATA.rglob("data.yaml"))
assert yaml_files, "zip 裡沒有 data.yaml：匯出時格式要選 YOLOv11 或 YOLOv8"
DATA = yaml_files[0].parent
print("資料集位置：", DATA)

## 4. 檢查資料集

Roboflow 的 `data.yaml` 裡的路徑常常是相對的，在 Colab 會找不到，這裡改寫成絕對路徑。
同時檢查類別順序、每一份資料的數量、每一類有幾個框。

In [ ]:
import yaml
from collections import Counter

cfg = yaml.safe_load((DATA / "data.yaml").read_text(encoding="utf-8"))
names = cfg["names"]
if isinstance(names, dict):                      # 有些版本寫成 {0: 'ped_green', ...}
    names = [names[i] for i in sorted(names)]
assert names == EXPECTED_CLASSES, f"類別順序不對：資料集是 {names}，應該是 {EXPECTED_CLASSES}。請在 Roboflow 改好類別名稱後重新匯出"

splits = {"train": "train", "val": "valid", "test": "test"}
for key, folder in splits.items():
    assert (DATA / folder / "images").is_dir(), f"少了 {folder}/images：Roboflow 匯出時要有 {folder} 這一份"

def count(folder):
    imgs = [p for p in (DATA / folder / "images").iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
    boxes, empty = Counter(), 0
    for img in imgs:
        label = DATA / folder / "labels" / (img.stem + ".txt")
        lines = label.read_text().splitlines() if label.exists() else []
        ids = [int(l.split()[0]) for l in lines if l.strip()]
        if not ids: empty += 1                   # 沒有框的照片（教模型「這不是號誌」）
        boxes.update(ids)
    return len(imgs), empty, boxes

print(f"{'資料':<6}{'照片':>6}{'沒框的':>8}" + "".join(f"{n:>12}" for n in EXPECTED_CLASSES))
stats = {}
for key, folder in splits.items():
    n, empty, boxes = count(folder)
    stats[key] = (n, empty, boxes)
    print(f"{folder:<6}{n:>6}{empty:>8}" + "".join(f"{boxes[i]:>12}" for i in range(len(EXPECTED_CLASSES))))

assert stats["test"][0] > 0, "測試資料是空的：請在 Roboflow 把 2～3 個沒用來訓練的路口放進 Test"
for i, name in enumerate(EXPECTED_CLASSES):
    if stats["train"][2][i] < 300:
        print(f"⚠ 訓練資料的 {name} 只有 {stats['train'][2][i]} 個框，建議至少 300 個，結果可能不穩定")
if stats["train"][1] < 0.1 * stats["train"][0]:
    print("⚠ 沒有框的照片不到一成：多加一些車尾燈、招牌、樹的照片，誤報才會少")

cfg.update(path=str(DATA), train="train/images", val="valid/images", test="test/images", names=EXPECTED_CLASSES, nc=len(EXPECTED_CLASSES))
DATA_YAML = DATA / "data_colab.yaml"
DATA_YAML.write_text(yaml.safe_dump(cfg, allow_unicode=True), encoding="utf-8")
print("已寫入", DATA_YAML)

## 5. 訓練

從 YOLO11n（COCO 預先訓練）開始微調。T4 GPU 上大約 30～60 分鐘，看照片數量。
訓練中會一直印出每一輪的分數；連續 20 輪沒有進步會自己停。

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(
    data=str(DATA_YAML), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, seed=SEED,
    deterministic=True, project="/content/runs", name="signals", exist_ok=True,
    # 紅綠之分就是這個模型的全部：色相只做很小的擾動（預設也很小，這裡寫出來讓報告看得到）
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    fliplr=0.5,        # 左右翻轉不會改變號誌或斑馬線的意思
)
RUN = Path("/content/runs/signals")
BEST = RUN / "weights" / "best.pt"
assert BEST.exists(), "訓練沒有產生 best.pt，請往上看錯誤訊息"

## 6. 用「測試路口」的照片評分

這些照片的路口完全沒出現在訓練資料裡，這裡的數字才是報告要用的。
**精確率（Precision）最重要**：說錯號誌比不說更危險。

In [ ]:
import csv

best = YOLO(str(BEST))
metrics = best.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, project="/content/runs", name="signals_test", exist_ok=True)

rows = []
for k, cls in enumerate(metrics.box.ap_class_index):
    rows.append({"類別": EXPECTED_CLASSES[int(cls)], "精確率 Precision": round(float(metrics.box.p[k]), 3),
                 "召回率 Recall": round(float(metrics.box.r[k]), 3), "mAP50": round(float(metrics.box.ap50[k]), 3)})
for name in set(EXPECTED_CLASSES) - {r["類別"] for r in rows}:
    print(f"⚠ 測試資料裡沒有 {name}，這一類沒有分數")

print(f"\n{'類別':<12}{'精確率':>8}{'召回率':>8}{'mAP50':>8}")
for r in rows:
    flag = "  ← 精確率低於 0.95，誤報偏多" if r["精確率 Precision"] < 0.95 else ""
    print(f"{r['類別']:<12}{r['精確率 Precision']:>8}{r['召回率 Recall']:>8}{r['mAP50']:>8}{flag}")

assert rows, "測試資料裡一個框都沒有，無法評分"
with open(RUN / "test_metrics.csv", "w", newline="", encoding="utf-8-sig") as f:   # utf-8-sig：Excel 打開中文不會亂碼
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader(); w.writerows(rows)

## 7. 轉成 App 用的 ONNX 檔並檢查

App 用的是 onnxruntime-web 1.20.1，這裡固定 opset 17、輸入大小固定，確保它讀得了。

In [ ]:
import numpy as np, onnxruntime as ort

onnx_path = Path(best.export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, dynamic=False))
final = RUN / OUTPUT_NAME
shutil.copy(onnx_path, final)

# 用 onnxruntime 實際跑一次假的畫面，確認輸入輸出形狀
sess = ort.InferenceSession(str(final), providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
out = sess.run(None, {inp.name: np.zeros((1, 3, IMGSZ, IMGSZ), dtype=np.float32)})[0]
anchors = sum((IMGSZ // s) ** 2 for s in (8, 16, 32))
print("輸入", inp.name, inp.shape, "→ 輸出", out.shape)
assert out.shape == (1, 4 + len(EXPECTED_CLASSES), anchors), f"輸出形狀不對，應該是 (1, {4 + len(EXPECTED_CLASSES)}, {anchors})"
print(f"✓ {final.name}：{final.stat().st_size / 1e6:.1f} MB，輸出 = 4 個框座標 + {len(EXPECTED_CLASSES)} 類分數 × {anchors} 個候選框")

## 8. 存到雲端硬碟並下載

存到雲端硬碟 `科展/訓練結果/日期時間/`：模型、測試分數、混淆矩陣、PR 曲線、訓練過程曲線（報告用）。

In [ ]:
import datetime
from google.colab import files

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
dest = Path(DRIVE_DIR) / "訓練結果" / stamp
dest.mkdir(parents=True, exist_ok=True)
for f in [final, RUN / "test_metrics.csv", RUN / "results.png", RUN / "results.csv", RUN / "args.yaml", BEST]:
    if f.exists(): shutil.copy(f, dest)
for f in Path("/content/runs/signals_test").glob("*.png"):     # 測試資料的混淆矩陣、PR 曲線
    shutil.copy(f, dest / ("test_" + f.name))
print("已存到雲端硬碟：", dest)
files.download(str(final))